In [ ]:
# Data preprocessing
import pandas                  as     pd
import numpy                   as     np
import missingno               as     msno

from   sklearn.model_selection import train_test_split
import statsmodels.api         as     sm
import scipy.stats             as     stats

# Visualization
import seaborn                 as     sns
import matplotlib.pyplot       as     plt
sns.set_style("whitegrid")

import warnings
warnings.filterwarnings("ignore")

In [ ]:
import random  # Set the global random seed
# Set the global random seed
def set_global_seed(seed_value):
    random.seed(seed_value)  # Set the Python random seed
    np.random.seed(seed_value)  # Set the NumPy random seed

set_global_seed(66)

In [ ]:
# Set the Matplotlib font to Times New Roman
plt.rcParams['font.family'] = 'Times New Roman'

In [ ]:
data = pd.read_csv('')

# Causal Graph Construction Using the DirectLiNGAM Algorithm

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
import lingam  # Ensure that the lingam package is installed
from lingam.utils import make_prior_knowledge, make_dot

In [ ]:
# -------------------------------
# 1. Construct the causal graph using DirectLiNGAM
# -------------------------------

# Helper function: convert the adjacency matrix into an edge list (showing only non-zero weighted edges)
def show(adjacency_matrix, var_names):
    n = adjacency_matrix.shape[0]
    result = []
    for i in range(n):
        for j in range(n):
            if adjacency_matrix[i, j] != 0:
                # Here, j -> i indicates that j is a parent node of dependent variable i
                result.append((var_names[j], var_names[i]))
    return result

# Specify prior knowledge: treat the selected variable as a sink variable (affected by other variables only)
prior_knowledge = make_prior_knowledge(
    n_variables=23,
    sink_variables=[22]  # Index 22 corresponds to the selected sink variable
)
print("Prior knowledge matrix:")
print(prior_knowledge)

# Construct the DirectLiNGAM model and fit it to the data; the input is a NumPy array
model = lingam.DirectLiNGAM(prior_knowledge=prior_knowledge)
model.fit(data.values)

print("Adjacency matrix learned by DirectLiNGAM:")
print(model.adjacency_matrix_)

# Extract causal edges from the adjacency matrix
edges = show(model.adjacency_matrix_, var_names=np.array(data.columns))


In [ ]:
# Draw the causal graph using NetworkX
G = nx.DiGraph()
G.add_edges_from(edges)
pos = nx.shell_layout(G)
plt.figure(figsize=(10, 10))
nx.draw(G, pos, node_size=800, node_color="#709BC8", with_labels=True, arrowsize=20)
plt.title("Causal graph constructed by DirectLiNGAM")
plt.show()

In [ ]:
edges

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
import lingam  # Ensure that the lingam package is installed
from lingam.utils import make_prior_knowledge
# -------------------------------
# 1. Construct the causal graph using DirectLiNGAM
# -------------------------------

# Helper function: convert the adjacency matrix into an edge list (showing only non-zero weighted edges)
def show(adjacency_matrix, var_names):
    n = adjacency_matrix.shape[0]
    result = []
    for i in range(n):
        for j in range(n):
            if adjacency_matrix[i, j] != 0:
                # Here, j -> i indicates that j is a parent node of dependent variable i
                result.append((var_names[j], var_names[i]))
    return result

# Specify prior knowledge: treat the selected variable as a sink variable (affected by other variables only)
prior_knowledge = make_prior_knowledge(
    n_variables=23,
    sink_variables=[22]
)
print("Prior knowledge matrix:")
print(prior_knowledge)

# Construct the DirectLiNGAM model and fit it to the data using a NumPy array
model = lingam.DirectLiNGAM(prior_knowledge=prior_knowledge)
model.fit(data.values)

print("Adjacency matrix learned by DirectLiNGAM:")
print(model.adjacency_matrix_)

# Extract causal edges from the adjacency matrix
edges = show(model.adjacency_matrix_, var_names=np.array(data.columns))

# -------------------------------
# 2. Preliminary visualization of the causal graph
# -------------------------------

plt.rcParams['font.family'] = 'Times New Roman'

G = nx.DiGraph()
G.add_edges_from(edges)

# Use a shell layout for a clearer graph presentation
pos = nx.shell_layout(G)

plt.figure(figsize=(12, 12))
nx.draw_networkx_nodes(G, pos,
                       node_size=1000,
                       node_color="#A1C9F4",
                       edgecolors="#3B4C6A",
                       linewidths=2)
nx.draw_networkx_edges(G, pos,
                       edge_color="#6B6B6B",
                       width=2,
                       arrows=True,
                       arrowsize=20)
nx.draw_networkx_labels(G, pos,
                        font_size=14,
                        font_color="#1C1C1C",
                        font_family='Times New Roman')

plt.title("Causal Graph Constructed by DirectLiNGAM", fontsize=18, fontweight='bold')
plt.axis('off')
plt.tight_layout()
plt.show()

# -------------------------------
# 3. Compute ISM layers
# -------------------------------

# Define the variable-name list (must match the column names in the CSV file)
variables = [
    'Daily sailing hours', 'Daily sailing distance', 'Total time', 'Total sailing distance',
    'Sea surface temperature at 2 meters', 'Main engine RPM', 'Atmospheric pressure',
    'Remaining distance to destination port', 'Overall average speed', 'Mean wave period',
    'Ship course', 'Relative swell direction angle', 'Main engine fuel consumption'
]

# Convert the adjacency matrix into a binary influence matrix
adjacency_matrix = model.adjacency_matrix_
influence_matrix = (adjacency_matrix != 0).astype(int)

# Method: compute layers based on topological sorting (suitable for DAGs)
def compute_ism_layers_topological(influence_matrix):
    n = influence_matrix.shape[0]
    in_degree = [0] * n
    for i in range(n):
        for j in range(n):
            if influence_matrix[i][j] == 1:
                in_degree[j] += 1
    layers = [0] * n
    queue = [i for i in range(n) if in_degree[i] == 0]
    
    while queue:
        node = queue.pop(0)
        for neighbor in range(n):
            if influence_matrix[node][neighbor] == 1:
                layers[neighbor] = max(layers[neighbor], layers[node] + 1)
                in_degree[neighbor] -= 1
                if in_degree[neighbor] == 0:
                    queue.append(neighbor)
    return layers

layers = compute_ism_layers_topological(influence_matrix)
# Output the layer of each variable
for i, var in enumerate(variables):
    print(f"{var}: Layer {layers[i]}")

# -------------------------------
# 4. Plot the optimized causal graph with ISM layering
# -------------------------------

# Construct the causal graph using column names from data.columns as node labels
var_names = np.array(data.columns)
G = nx.DiGraph()
edges = show(adjacency_matrix, var_names)
G.add_edges_from(edges)

# Define a custom layer-based layout function to keep nodes within each layer aligned
def hierarchical_layout(G, layers, layer_sep=2, node_sep=1):
    # Collect nodes in each layer (assuming each node name can be matched to an index in var_names)
    layer_nodes = {}
    for node in G.nodes():
        idx = list(var_names).index(node)
        l = layers[idx]
        layer_nodes.setdefault(l, []).append(node)
    pos = {}
    for l in sorted(layer_nodes.keys()):
        nodes = layer_nodes[l]
        y = -l * layer_sep  # y-axis coordinate; higher layer values are placed lower for a top-down visualization
        width = (len(nodes) - 1) * node_sep
        for i, node in enumerate(sorted(nodes)):
            x = i * node_sep - width / 2
            pos[node] = (x, y)
    return pos

# Generate the custom hierarchical layout
pos = hierarchical_layout(G, layers)

# -------------------------------

In [ ]:
# Define a refined custom color map
# -------------------------------

import matplotlib.colors as mcolors

# Define a restrained and refined color palette (extend as needed)
custom_color_list = [
    "#FFB3BA",  # Light pink
    "#FFDFBA",  # Light orange
    "#FFFFBA",  # Light yellow
    "#BAFFC9",  # Light green
    "#BAE1FF",  # Light blue
    "#B5EAD7",  # Soft mint green
    "#C9C9FF",  # Light lavender blue
    "#F1E3FF",  # Light lavender
    "#FFD1DC",  # Light peach pink
    "#FFECB3",  # Light amber
    "#E1F5FE"   # Light sky blue
]
max_layer = max(layers)
# Note: N is set to max_layer + 1 so that each layer maps to a discrete color
custom_cmap = mcolors.LinearSegmentedColormap.from_list('custom', custom_color_list, N=max_layer + 1)

node_colors = []
# Assign node colors by layer; node order follows data.columns and corresponds to the layers array
for node in G.nodes():
    idx = list(var_names).index(node)
    # custom_cmap returns RGBA tuples
    node_colors.append(custom_cmap(layers[idx]))

plt.figure(figsize=(12, 12))
nx.draw_networkx_nodes(G, pos,
                       node_size=1000,
                       node_color=node_colors,
                       edgecolors="#3B4C6A",
                       linewidths=2)
nx.draw_networkx_edges(G, pos,
                       edge_color="#6B6B6B",
                       width=2,
                       arrows=True,
                       arrowsize=20)
nx.draw_networkx_labels(G, pos,
                        font_size=14,
                        font_color="#1C1C1C",
                        font_family='Times New Roman')

plt.title("Optimized Causal Graph with ISM Layering", fontsize=18, fontweight='bold')
plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
print("\nNode-to-node connections (parent node -> child node):")
for parent, child in edges:
    print(f"{parent} -> {child}")